## Urban Vehicle Residency and Parking Supply Analysis

UP213 Urban Data Science SQ25

Group Members: Simon Han, Tomohiro Ujikawa

### Objective
Our project aims to address vehicle living in LA by identifying where vehicle residents are clustered and where there could be safe and sanctioned places to park at night, particularly in areas with excess capacity through city parking garages.

### Background
Vehicle residency is rising among working-class individuals and families as housing costs outpace wages in Los Angeles. Safe parking programs offer legal overnight parking for vehicle residents along with access to restrooms and supportive services. However, the transient and discreet nature of many vehicle residents poses operational challenges for these programs. In addition, long waitlists and limited capacity at individual sites often discourage those with urgent needs from enrolling. To address these gaps, it is essential to better align areas of high need with high-capacity parking facilities that can operate with minimal staffing and oversight. 

### Key Questions
- Where are the vehicle homeless **clusters** in Los Angeles?
- Where are parking facilities with high **capacity** in areas with high need?

### Dataset
- LAHSA Homeless Count: vehicle dwelling counts by tract
- ACS: poverty, rent burden, vehicle ownership, housing overcrowding, etc.
- TIGER/Line: Census tract boundaries for spatial analysis
- SCAG 2019 Regional Land Use Type: public parking facilities (LU19 code 1247)
- Public Facilities Data: Locations of essential services (hospitals, schools, transportation hubs, and food pantries)

### Methods
- **Spatial Joins** to overlay homelessness data with garage locations
- **Random Forests** to predict vehicle homelessness based on demographic and economic factors
- **Cluster Analysis** to identify different typologies of vehicle homelessness
- **Visualization** of results through various maps and charts

In [1]:
import os
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn import preprocessing, metrics
import contextily as ctx
import requests

os.getcwd()

'/Users/admin/Documents/GitHub/UP213_group'

## Data Preparation

In [2]:
# Load TIGER/Line shapefiles for LA County
for year in range(2019, 2025):
    path = f"Data/tl_shp_library/tl_{year}_06_tract.zip"
    tracts = gpd.read_file(path)
    la_tracts = tracts[(tracts["COUNTYFP"] == "037") & (tracts["STATEFP"] == "06")]
    la_tracts = la_tracts.set_index('GEOID')
    
    var_name = f"tracts{str(year)[-2:]}"
    globals()[var_name] = la_tracts

print("Loaded census tract boundaries for 2019-2024")

Loaded census tract boundaries for 2019-2024


In [3]:
# Load land use and parking data
luGdf = gpd.read_file("Data/landuse.gpkg")
luGdf['LU19'] = pd.to_numeric(luGdf['LU19'], errors='coerce')

# Extract parking areas (LU19 code 1247)
Parking = luGdf[luGdf['LU19'] == 1247].to_crs(epsg=3497)
print(f"Found {len(Parking)} parking parcels")

DataSourceError: Data/landuse.gpkg: No such file or directory

In [ ]:
# Load public facilities
ArtsRecreation = gpd.read_file("data/public_facilities/ArtsRecreation.gpkg").to_crs(epsg=3497)
Education = gpd.read_file("data/public_facilities/Education.gpkg").to_crs(epsg=3497)
Hospitals = gpd.read_file("data/public_facilities/Hospitals.gpkg").to_crs(epsg=3497)
Transportation = gpd.read_file("data/public_facilities/Transportation.gpkg").to_crs(epsg=3497)

# Create 2km buffers around tracts for amenity counting
tracts23_buffer = tracts23.to_crs(epsg=3497)
tracts23_buffer["geometry"] = tracts23_buffer.buffer(2000)

In [ ]:
# Function to count facilities within tract buffers
def count_facilities_within_tracts(tracts_gdf, points_gdf, facility_name):
    tmp = gpd.sjoin(tracts_gdf, points_gdf, predicate='intersects')
    counts = tmp.groupby('GEOID').size()
    counts.name = f'n_{facility_name}'
    tracts_gdf = tracts_gdf.join(counts)
    tracts_gdf.fillna({f'n_{facility_name}': 0}, inplace=True)
    return tracts_gdf

# Count facilities
tracts23_buffer = count_facilities_within_tracts(tracts23_buffer, ArtsRecreation, 'ArtsRecreation')
tracts23_buffer = count_facilities_within_tracts(tracts23_buffer, Education, 'Education')
tracts23_buffer = count_facilities_within_tracts(tracts23_buffer, Hospitals, 'Hospitals')
tracts23_buffer = count_facilities_within_tracts(tracts23_buffer, Transportation, 'Transportation')

# Count parking
tmpgdf_parking = gpd.sjoin(tracts23_buffer, Parking, predicate='intersects')
Parking_counts = tmpgdf_parking.groupby('GEOID_left').size()
Parking_counts.name = 'n_Parking'
tracts23_buffer = tracts23_buffer.set_index('GEOID').join(Parking_counts).reset_index()
tracts23_buffer['n_Parking'] = tracts23_buffer['n_Parking'].fillna(0).astype(int)

print("Amenity counting complete")

In [ ]:
# Process LAHSA homeless count data for 2022-2024
representative_cols = ['year', 'lacity', 'spa', 'sd', 'cd']

for year in [22, 23, 24]:
    hc = pd.read_csv(f"Data/LAHSA/streetcount{year}_original.csv")
    hc.columns = hc.columns.str.lower()
    
    # Extract base tract from split tracts
    hc['tract_split_base'] = hc['tract_split'].astype(str).str.extract(r'(\d{6})')
    
    # Aggregate split tracts
    numeric_cols = hc.select_dtypes(include='number').columns.tolist()
    sum_cols = [col for col in numeric_cols if col not in representative_cols]
    
    hc_agg = hc.groupby('tract_split_base').agg(
        {**{col: 'first' for col in representative_cols},
         **{col: 'sum' for col in sum_cols}}
    ).reset_index()
    
    hc_agg['GEOID'] = '06037' + hc_agg['tract_split_base']
    hc_agg = hc_agg.set_index('GEOID')
    globals()[f"hc{year}"] = hc_agg

# Load 2020 data separately
hc20 = pd.read_csv('Data/LAHSA/streetcount20.csv', dtype={'tract': str})
hc20['GEOID'] = '06037' + hc20['tract']
hc20 = hc20.set_index('GEOID')
hc20.columns = hc20.columns.str.lower()

print("LAHSA data processed")

In [ ]:
# Load ACS demographic data
acs_vars = [
    "B25077_001E",  # Median home value
    "B25064_001E",  # Median gross rent
    "B25070_006E", "B25070_007E", "B25070_001E",  # Rent burden
    "B25091_007E", "B25091_008E", "B25091_001E",  # Owner cost burden
    "B23025_005E", "B23025_003E",  # Unemployment
    "B17001_002E", "B17001_001E",  # Poverty
    "B03003_003E", "B03003_001E",  # Hispanic population
    "B25014_003E", "B25014_004E", "B25014_001E"  # Overcrowding
]

key = 'xx'
years = [2020, 2022, 2023]

for year in years:
    var_string = ",".join(acs_vars)
    url = f'https://api.census.gov/data/{year}/acs/acs5?get={var_string}&for=tract:*&in=state:06 county:037&key={key}'
    
    r = requests.get(url)
    data = r.json()
    df = pd.DataFrame(data[1:], columns=data[0])
    
    df['GEOID'] = '06037' + df['tract']
    df.set_index('GEOID', inplace=True)
    df.drop(columns=['state', 'county', 'tract'], inplace=True)
    
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    
    df = df.applymap(lambda x: np.nan if isinstance(x, (int, float)) and x < 0 else x)
    globals()[f'acs{str(year)[-2:]}'] = df

print("ACS data loaded")

## Analysis

In [ ]:
# Merge datasets
merged20 = acs20.join(hc20, how='left', rsuffix='_hc')
merged22 = acs22.join(hc22, how='left', rsuffix='_hc') 
merged23 = acs23.join(hc23, how='left', rsuffix='_hc')

# Add year indicators
merged20["year"] = 2020
merged22["year"] = 2022
merged23["year"] = 2023

# Combine years
merged_all = pd.concat([merged20.reset_index(), 
                        merged22.reset_index(), 
                        merged23.reset_index()], axis=0).reset_index(drop=True)

# Add amenity counts
tracts23 = tracts23.reset_index()
cols_to_merge = ['GEOID', 'n_ArtsRecreation', 'n_Education', 'n_Hospitals', 
                 'n_Transportation', 'n_Parking']
tract_counts = tracts23[cols_to_merge].copy()
merged_all = merged_all.merge(tract_counts, on="GEOID", how="left")

# Add geometry
geometry23 = tracts23[["GEOID", "geometry"]]
merged_all = merged_all.merge(geometry23, on="GEOID", how="left")
merged_all = gpd.GeoDataFrame(merged_all, geometry="geometry", crs=4326)

# Create variables
merged_all['total_vehicle'] = merged_all['totcars'] + merged_all['totvans'] + merged_all['totcampers']
merged_all['total_unsheltered'] = merged_all['total_vehicle'] + merged_all['tottents'] + merged_all['totencamp']

# Add centroids
centroids = merged_all.geometry.centroid
merged_all["lon"] = centroids.x
merged_all["lat"] = centroids.y

print(f"Final dataset: {len(merged_all)} rows")

In [ ]:
# Machine Learning: Predict Vehicle Homelessness
feature_vars = ['year', 'n_ArtsRecreation', 'n_Education', 'n_Hospitals', 
                'n_Transportation', 'n_Parking', 'lon', 'lat'] + acs_vars
target_var = 'total_vehicle'

# Clean dataset
df_to_fit = merged_all[feature_vars + [target_var]].dropna()
print(f"Training data: {len(df_to_fit)} observations")

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    df_to_fit[feature_vars], df_to_fit[target_var], test_size=0.25, random_state=1)

# Train model
rf = RandomForestRegressor(n_estimators=50, random_state=1)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

print(f'Mean Absolute Error: {metrics.mean_absolute_error(y_test, y_pred):.2f}')
print(f'R² Score: {metrics.r2_score(y_test, y_pred):.3f}')

In [ ]:
# Feature importance analysis
importances = rf.feature_importances_
forest_importances = pd.Series(importances, index=X_train.columns)
forest_importances.sort_values(ascending=False, inplace=True)

# Plot top 10
fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(x=forest_importances.values[:10], y=forest_importances.index[:10], ax=ax)
ax.set_title("Top 10 Predictors of Vehicle Homelessness")
ax.set_xlabel("Feature Importance")
plt.tight_layout()
plt.show()

print("\nTop 5 Predictors:")
for i, (feature, importance) in enumerate(forest_importances.head(5).items(), 1):
    print(f"{i}. {feature}: {importance:.4f}")

In [ ]:
# Cluster Analysis
top_features = forest_importances.index[:10]
scaler = preprocessing.StandardScaler().fit(merged_all[top_features])
merged_all_scaled = pd.DataFrame(
    scaler.transform(merged_all[top_features]), 
    columns=top_features, 
    index=merged_all.index
).dropna()

# K-means clustering
kmeans = KMeans(n_clusters=5, random_state=0).fit(merged_all_scaled)
merged_all_scaled['cluster_id'] = kmeans.labels_
merged_all['cluster_id'] = merged_all_scaled['cluster_id']

print("Cluster sizes:")
print(merged_all_scaled.groupby('cluster_id').size())

## Visualization

In [ ]:
# Map vehicle homelessness (2023)
merged23_map = merged_all[merged_all['year'] == 2023].copy()

fig, ax = plt.subplots(figsize=(12, 10))
merged23_map.to_crs('EPSG:3857').plot(
    column='total_vehicle', cmap='YlOrRd', legend=True, ax=ax, alpha=0.7)
ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron, zoom=12)
ax.set_title('Vehicle Homelessness by Census Tract, 2023', fontsize=14)
ax.set_ylim([3.98e6, 4.14e6])
ax.set_xticks([])
ax.set_yticks([])
plt.tight_layout()
plt.show()

In [ ]:
# Map parking supply
fig, ax = plt.subplots(figsize=(12, 10))
tracts23.to_crs(epsg=3857).plot(
    column='n_Parking', cmap='Blues', legend=True, ax=ax, alpha=0.7)
ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron, zoom=12)
ax.set_title("Parking Parcels within 2000m of Each Census Tract", fontsize=14)
ax.set_ylim([3.98e6, 4.14e6])
ax.set_xticks([])
ax.set_yticks([])
plt.tight_layout()
plt.show()

In [ ]:
# Model validation plot
fig, ax = plt.subplots(figsize=(8, 6))
sns.regplot(x=y_test, y=y_pred, ax=ax, scatter_kws={'alpha':0.6})
ax.set_xlabel('Actual Vehicle Homelessness')
ax.set_ylabel('Predicted Vehicle Homelessness')
ax.set_title('Model Predictions vs Actual Values')

# Perfect prediction line
min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())
ax.plot([min_val, max_val], [min_val, max_val], 'r--', alpha=0.8, label='Perfect Prediction')
ax.legend()
plt.tight_layout()
plt.show()

## Key Findings

### Top Predictors of Vehicle Homelessness:
1. **Housing Cost Burden**: Areas where owner costs exceed 40% of income
2. **Median Home Value**: Higher property values correlate with vehicle homelessness
3. **Hispanic/Latino Population**: Strong demographic predictor
4. **Renter Concentration**: Areas with high renter populations

### Spatial Patterns:
- Vehicle homelessness clusters in high-cost areas
- Geographic concentration enables targeted interventions
- Variation in parking supply across census tracts

### Policy Implications:
- **Target high-burden areas**: Focus on census tracts with housing cost burden >40%
- **Leverage existing infrastructure**: Utilize underused parking garages
- **Service accessibility**: Ensure proximity to restrooms, food, transportation
- **Predictive planning**: Use demographic trends to anticipate future need

### Recommendations:
1. Prioritize safe parking programs in identified high-need clusters
2. Conduct detailed site assessments of parking facilities in target areas
3. Develop partnerships with property owners and service providers
4. Implement monitoring system to track program effectiveness

In [ ]:
# Export results
merged_all.to_file("Data/vehicle_homelessness_analysis.gpkg", driver="GPKG")
merged_all.drop(columns='geometry').to_csv("Data/vehicle_homelessness_analysis.csv", index=False)

print("Analysis complete!")
print("Data exported to:")
print("- Data/vehicle_homelessness_analysis.gpkg (with geometry)")
print("- Data/vehicle_homelessness_analysis.csv (tabular data)")